# Phase 1 — Thorough Analysis

Deep analysis of a trained meta-encoder across three dimensions:

| Section | Content |
|---------|---------|
| 3. Circuit Discovery | C3 elevation · C4 diversity · C5 purity · circuit viewer · span heatmap |
| 4. Architecture Transfer | Compare C1/C2 across ResNet18 / 34 / 50 |
| 5. Dataset Generalization | Evaluate C1/C2 on CIFAR-10 / CIFAR-100 / STL-10 |

**Prerequisites:** a trained checkpoint from `nb01_training_and_validation.ipynb`.

**Next step:** use `nb03_causal_interventions.ipynb` to test whether the discovered circuits are causally useful and behaviorally meaningful.

## 0. Colab Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jacobposchl/trainable-circuits'
REPO_DIR = '/content/model_interpretability'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print(f'Repo:     {REPO_DIR}')
print(f'Data:     {DATA_DIR}')
print(f'Checkpts: {CKPT_DIR}')


## 1. Configuration

Edit **only this cell** to switch the model being analyzed.

In [ ]:
# -----------------------------------------------------------------------------
#  SELECT MODEL  —  uncomment one line
# -----------------------------------------------------------------------------
MODEL_CONFIG = CONFIG_DIR + '/models/resnet18.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet34.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet50.yaml'

print(f'Using config: {MODEL_CONFIG}')


## 2. Load Model & Representations

In [ ]:
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from evaluation.circuit_analysis import CircuitAnalyzer, load_checkpoint
from data.cifar import get_standard_loaders

with open(MODEL_CONFIG) as f:
    config = yaml.safe_load(f)

config['data']['data_dir']          = DATA_DIR
config['logging']['checkpoint_dir'] = CKPT_DIR + '/' + config['experiment']['name']

checkpoint_path = config['logging']['checkpoint_dir'] + '/best.pt'
device          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone, meta_encoder, info_loss = load_checkpoint(config, checkpoint_path, device)

_, val_loader = get_standard_loaders(
    data_dir    = config['data']['data_dir'],
    batch_size  = config['data'].get('batch_size', 256),
    num_workers = config['data'].get('num_workers', 4),
    augment     = False,
)


In [ ]:
MAX_SAMPLES = 2000

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
print(f'Collecting representations (max {MAX_SAMPLES} samples)...')
data = analyzer.collect_representations(max_samples=MAX_SAMPLES)

flow_targets = data['flow_targets']
z_list       = data['z_list']
labels       = data['labels']
images       = data['images']
z_list_np    = [z.numpy() for z in z_list]
labels_np    = labels.numpy()
N = labels.shape[0]
L = len(z_list)
print(f'Collected {N} samples across {L} layers.')


## 3. Circuit Discovery — Criteria 3–5

Span-centric image-centric pipeline: for each contiguous span `[l_start, l_end]`, concatenate
z-vectors, reduce with UMAP (cosine), cluster with HDBSCAN, keep canonical clusters (1%–40% of images).

In [ ]:
from evaluation import SpanCentricDiscovery

disc_cfg  = config.get('discovery', {})
discovery = SpanCentricDiscovery(
    n_layers             = L,
    umap_n_components    = disc_cfg.get('umap_n_components', 15),
    umap_n_neighbors     = disc_cfg.get('umap_n_neighbors', 15),
    min_cluster_fraction = disc_cfg.get('min_cluster_fraction', 0.01),
    max_cluster_fraction = disc_cfg.get('max_cluster_fraction', 0.40),
    min_cluster_size     = disc_cfg.get('hdbscan_min_cluster_size', 5),
)

print(f'Running discovery across {L*(L+1)//2} candidate spans...')
circuits = discovery.discover_all(z_list_np)
print(f'Found {len(circuits)} canonical circuits.')


In [ ]:
from evaluation import within_span_elevation

population_sims = discovery.compute_span_similarities(z_list_np, span=(0, L - 1))

for c in circuits:
    cluster_sims = discovery.compute_span_similarities(
        z_list_np, span=c['span'], image_mask=c['image_mask']
    )
    c.update(within_span_elevation(cluster_sims, population_sims))
    c['purity']   = SpanCentricDiscovery.compute_class_purity(c, labels_np)
    c['span_len'] = c['span'][1] - c['span'][0] + 1

n_pass_c3 = sum(1 for c in circuits if c.get('passes', False))
print(f'C3 — Elevation: {n_pass_c3}/{len(circuits)} circuits pass (s = 1.0)')


In [ ]:
elevations = [c['elevation_sigma'] for c in circuits]
if elevations:
    print(f'Elevation range : {min(elevations):.2f}s  —  {max(elevations):.2f}s')
    print(f'Mean elevation  : {sum(elevations)/len(elevations):.2f}s')
    print(f'C3 Status       : {"PASS ?" if n_pass_c3 > 0 else "FAIL ?"}')
else:
    print('No circuits found.')


In [ ]:
from evaluation import circuit_diversity, plot_span_coverage

div = circuit_diversity([c['span'] for c in circuits], L)
print(f'Layer coverage : {div["coverage"]:.1%}  (target = 60%)')
print(f'Circuits found : {div["n_circuits"]}')
print(f'C4 Status      : {"PASS ?" if div["passes"] else "FAIL ?"}')

fig = plot_span_coverage(circuits, L)
plt.show()


In [ ]:
from evaluation import class_purity_distribution

purities   = [c['purity'] for c in circuits]
pur_result = class_purity_distribution(purities)

print(f'Agnostic  (<0.3) : {pur_result["n_agnostic"]}')
print(f'Mixed  (0.3-0.7) : {pur_result["n_middle"]}')
print(f'Specific  (>0.7) : {pur_result["n_specific"]}')
print(f'C5 Status        : {"PASS ?" if pur_result["passes"] else "FAIL ?"}')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(purities, bins=20, range=(0, 1), edgecolor='black', alpha=0.75)
ax.axvline(0.3, color='green', linestyle='--', linewidth=1.2, label='Agnostic threshold')
ax.axvline(0.7, color='red',   linestyle='--', linewidth=1.2, label='Specific threshold')
ax.set_xlabel('Class purity')
ax.set_ylabel('Number of circuits')
ax.set_title('C5 — Class Purity Distribution')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
from evaluation.circuit_viz import plot_multi_circuit_histogram

membership = SpanCentricDiscovery.multi_circuit_membership(circuits, n_images=N)
print(f'Mean memberships : {membership.mean():.1f}')
print(f'Max memberships  : {membership.max()}')
fig = plot_multi_circuit_histogram(membership)
plt.show()


In [ ]:
from evaluation.circuit_analysis import denormalize
from evaluation.circuit_viz import plot_circuit_members

def show_circuits(circuits, sort_by='elevation_sigma', n_show=3, n_images=12):
    """Display top circuits sorted by a metric, with per-image z-similarity profiles."""
    ranked = sorted([c for c in circuits if sort_by in c], key=lambda c: c[sort_by], reverse=True)[:n_show]
    if not ranked:
        print(f'No circuits with field {sort_by!r} found.')
        return
    for i, circuit in enumerate(ranked):
        mask = circuit['image_mask']
        l_s, l_e = circuit['span']
        K = mask.sum()
        profiles = np.zeros((K, L))
        for l in range(l_s, l_e + 1):
            zc  = z_list_np[l][mask]
            sim = zc @ zc.T
            np.fill_diagonal(sim, 0)
            profiles[:, l] = sim.sum(1) / max(K - 1, 1)
        torch_mask = torch.from_numpy(mask)
        imgs = denormalize(images[torch_mask]).numpy()[:n_images]
        lbs  = labels_np[mask][:n_images]
        prof = profiles[:n_images]
        val  = circuit.get(sort_by, float('nan'))
        fig  = plot_circuit_members(imgs, lbs, prof, circuit['span'], n_show=n_images)
        fig.suptitle(f'Circuit {i+1} | span=[L{l_s+1}–L{l_e+1}] | size={circuit["size"]} | {sort_by}={val:.2f}', fontsize=10, y=1.01)
        plt.show()


In [ ]:
print('Top circuits by elevation (most cohesive):')
show_circuits(circuits, sort_by='elevation_sigma', n_show=3)


In [ ]:
print('Class-specific circuits (purity > 0.7):')
show_circuits([c for c in circuits if c.get('purity', 0) > 0.7], sort_by='elevation_sigma', n_show=2)

print('Class-agnostic circuits (purity < 0.3):')
show_circuits([c for c in circuits if c.get('purity', 1) < 0.3], sort_by='elevation_sigma', n_show=2)


In [ ]:
from evaluation import plot_span_heatmap

for metric in ('elevation_sigma', 'purity', 'size'):
    fig = plot_span_heatmap(circuits, n_layers=L, metric=metric)
    plt.show()


## 4. Architecture Transfer

Compares C1 (R²) and C2 (Mean ?) across backbone depths.
Add entries to `ARCH_EXPERIMENTS` after training the corresponding checkpoints.

In [ ]:
# -----------------------------------------------------------------------------
#  ARCHITECTURE EXPERIMENTS  —  uncomment entries as checkpoints become available
# -----------------------------------------------------------------------------
ARCH_EXPERIMENTS = {
    'resnet18': CONFIG_DIR + '/models/resnet18.yaml',
    # 'resnet34': CONFIG_DIR + '/models/resnet34.yaml',
    # 'resnet50': CONFIG_DIR + '/models/resnet50.yaml',
}


In [ ]:
from evaluation import geometric_consistency
from evaluation.metrics import rich_profile_reconstruction_r2

arch_results = {}

for arch_name, arch_cfg_path in ARCH_EXPERIMENTS.items():
    with open(arch_cfg_path) as f:
        arch_cfg = yaml.safe_load(f)
    arch_cfg['data']['data_dir']          = DATA_DIR
    arch_cfg['logging']['checkpoint_dir'] = CKPT_DIR + '/' + arch_cfg['experiment']['name']
    arch_ckpt = arch_cfg['logging']['checkpoint_dir'] + '/best.pt'

    if not os.path.exists(arch_ckpt):
        print(f'{arch_name}: checkpoint not found ? skipping.')
        continue

    print(f'Evaluating {arch_name}...')
    bb, me, il = load_checkpoint(arch_cfg, arch_ckpt, device)
    _, vl = get_standard_loaders(data_dir=arch_cfg['data']['data_dir'], batch_size=arch_cfg['data'].get('batch_size', 256), num_workers=arch_cfg['data'].get('num_workers', 4), augment=False)
    az = CircuitAnalyzer(bb, me, vl, device)
    d  = az.collect_representations(max_samples=2000)
    n_a, l_a = d['labels'].shape[0], len(d['z_list'])
    i_a, i_b = torch.triu_indices(n_a, n_a, offset=1)
    if i_a.shape[0] > 50_000:
        perm = torch.randperm(i_a.shape[0])[:50_000]
        i_a, i_b = i_a[perm], i_b[perm]
    true_np = CircuitAnalyzer.compute_pair_profiles(d['flow_targets'], i_a, i_b).numpy()
    pred_list, true_list = [], []
    with torch.no_grad():
        for l in range(l_a):
            z_product = (d['z_list'][l][i_a] * d['z_list'][l][i_b]).to(device)
            pred_list.append(il.regressors[l](z_product).cpu().numpy())
            true_list.append((d['flow_targets'][l][i_a] * d['flow_targets'][l][i_b]).numpy())
    r2_res = rich_profile_reconstruction_r2(pred_list, true_list)
    z_sims = np.zeros((i_a.shape[0], l_a))
    for l in range(l_a):
        z_a = d['z_list'][l][i_a].numpy()
        z_b = d['z_list'][l][i_b].numpy()
        z_sims[:, l] = (z_a * z_b).sum(axis=1)
    gc_res = geometric_consistency(z_sims, true_np, l_a)
    arch_results[arch_name] = {'r2': r2_res['r2'], 'mean_rho': gc_res['mean_rho']}
    print(f'  R?={r2_res["r2"]:.4f}   Mean ?={gc_res["mean_rho"]:.4f}')


In [ ]:
if arch_results:
    names = list(arch_results)
    r2s   = [arch_results[n]['r2'] for n in names]
    rhos  = [arch_results[n]['mean_rho'] for n in names]
    x = np.arange(len(names)); w = 0.35
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - w/2, r2s,  w, label='R²', color='steelblue', edgecolor='black')
    ax.bar(x + w/2, rhos, w, label='Mean ?', color='seagreen', edgecolor='black')
    ax.axhline(0.70, color='steelblue', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.axhline(0.65, color='seagreen', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
    ax.set_title('Architecture Transfer — C1 (R²) and C2 (Mean ?)')
    ax.legend(); fig.tight_layout(); plt.show()
else:
    print('No results to plot. Train additional architectures first.')


## 5. Dataset Generalization

Evaluates the CIFAR-10 trained model on other datasets without retraining.
All images are resized to 32×32 to match the backbone's input resolution.

In [ ]:
# -----------------------------------------------------------------------------
#  DATASETS  —  uncomment entries to include in the comparison
# -----------------------------------------------------------------------------
DATASETS = [
    'cifar10',
    # 'cifar100',
    # 'stl10',
]


In [ ]:
from data.datasets import get_loaders as get_dataset_loaders
from evaluation.metrics import rich_profile_reconstruction_r2

dataset_results = {}

for ds_name in DATASETS:
    print(f'Evaluating on {ds_name}...')
    try:
        _, vl = get_dataset_loaders(dataset=ds_name, data_dir=DATA_DIR, batch_size=config['data'].get('batch_size', 256), num_workers=config['data'].get('num_workers', 4), augment=False)
        az = CircuitAnalyzer(backbone, meta_encoder, vl, device)
        d  = az.collect_representations(max_samples=2000)
        n_d, l_d = d['labels'].shape[0], len(d['z_list'])
        i_a, i_b = torch.triu_indices(n_d, n_d, offset=1)
        if i_a.shape[0] > 50_000:
            perm = torch.randperm(i_a.shape[0])[:50_000]
            i_a, i_b = i_a[perm], i_b[perm]
        true_np = CircuitAnalyzer.compute_pair_profiles(d['flow_targets'], i_a, i_b).numpy()
        pred_list, true_list = [], []
        with torch.no_grad():
            for l in range(l_d):
                z_product = (d['z_list'][l][i_a] * d['z_list'][l][i_b]).to(device)
                pred_list.append(info_loss.regressors[l](z_product).cpu().numpy())
                true_list.append((d['flow_targets'][l][i_a] * d['flow_targets'][l][i_b]).numpy())
        r2_res = rich_profile_reconstruction_r2(pred_list, true_list)
        z_sims = np.zeros((i_a.shape[0], l_d))
        for l in range(l_d):
            z_a = d['z_list'][l][i_a].numpy()
            z_b = d['z_list'][l][i_b].numpy()
            z_sims[:, l] = (z_a * z_b).sum(axis=1)
        gc_res = geometric_consistency(z_sims, true_np, l_d)
        dataset_results[ds_name] = {'r2': r2_res['r2'], 'mean_rho': gc_res['mean_rho']}
        print(f'  R?={r2_res["r2"]:.4f}   Mean ?={gc_res["mean_rho"]:.4f}')
    except Exception as exc:
        print(f'  Skipping {ds_name}: {exc}')


In [ ]:
if dataset_results:
    names = list(dataset_results)
    r2s   = [dataset_results[n]['r2'] for n in names]
    rhos  = [dataset_results[n]['mean_rho'] for n in names]
    x = np.arange(len(names)); w = 0.35
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - w/2, r2s,  w, label='R²', color='steelblue', edgecolor='black')
    ax.bar(x + w/2, rhos, w, label='Mean ?', color='seagreen', edgecolor='black')
    ax.axhline(0.70, color='steelblue', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.axhline(0.65, color='seagreen', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
    ax.set_title('Dataset Generalization — C1 (R²) and C2 (Mean ?)')
    ax.legend(); fig.tight_layout(); plt.show()
else:
    print('No results to plot.')


## Next Step

Use `nb03_causal_interventions.ipynb` to test whether the circuits discovered here are causally useful and behaviorally meaningful.